# CLARION — ScamRadar Training Notebook
## DistilBERT Multilingual Scam Text Classifier (9 Classes)

**CLARION Threat Intelligence Platform**

This notebook fine-tunes `distilbert-base-multilingual-cased` to classify text descriptions
of suspicious calls/messages into 9 categories.

### Classes:
0. `legitimate` — normal, non-scam interaction
1. `fake_cbi` — Fake CBI digital arrest scam
2. `fake_ed` — Fake ED money laundering scam
3. `customs_parcel` — Customs/drug parcel scam
4. `court_summons` — Fake court summons scam
5. `trai_suspension` — TRAI number suspension scam
6. `rbi_freeze` — Fake RBI account freeze scam
7. `narcotics_bureau` — Fake NCB narcotics scam
8. `courier_scam` — Fake courier/delivery fraud

### What this does:
- All training data is pre-defined in Cell 2 — no uploads needed!
- Fine-tunes DistilBERT with HuggingFace Trainer API
- Evaluates and prints per-class F1 scores
- Downloads the model as a zip file

> ⏱️ Estimated time: ~30-45 minutes on Colab GPU (T4)

In [ ]:
# ── Cell 1: Install Dependencies ──────────────────────────────────────────────
!pip install -q transformers==4.41.2 datasets==2.19.2 torch scikit-learn evaluate accelerate
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Training Data ─────────────────────────────────────────────────────
# 1350+ labeled examples across 9 classes.
# Each example is a realistic description of a call/message.

RAW_DATA = [
  # ── legitimate (class 0) ────────────────────────────────────────────────────
  {'text': 'My bank sent me an OTP to confirm a transaction I made online', 'label': 0},
  {'text': 'The delivery agent called to confirm my address for the package', 'label': 0},
  {'text': 'I received a call from my credit card company about an offer', 'label': 0},
  {'text': 'My mobile operator sent an SMS about a new plan upgrade', 'label': 0},
  {'text': 'The bank called to verify a large transaction I made yesterday', 'label': 0},
  {'text': 'Got a call from insurance company for policy renewal reminder', 'label': 0},
  {'text': 'My employer deposited salary and sent me a confirmation SMS', 'label': 0},
  {'text': 'The gas agency called to schedule a cylinder delivery', 'label': 0},
  {'text': 'Received an email from IRCTC about my confirmed train ticket', 'label': 0},
  {'text': 'My friend transferred money to me and sent a payment link', 'label': 0},
  {'text': 'The landlord called about the rent agreement renewal', 'label': 0},
  {'text': 'My school sent a message about upcoming exam dates', 'label': 0},
  {'text': 'The hospital sent a reminder for my appointment tomorrow', 'label': 0},
  {'text': 'Got an SMS from my bank about successful net banking login', 'label': 0},
  {'text': 'My mutual fund company sent a statement by email', 'label': 0},

  # ── fake_cbi (class 1) ───────────────────────────────────────────────────────
  {'text': 'A person called saying he is a CBI officer and my bank account is linked to money laundering', 'label': 1},
  {'text': 'Someone claiming to be CBI called and said I am under digital arrest for financial fraud', 'label': 1},
  {'text': 'A caller said CBI has issued a warrant for my arrest and I need to stay on video call', 'label': 1},
  {'text': 'Person said they are from CBI and my Aadhaar was found in a criminal case', 'label': 1},
  {'text': 'Got a call from someone claiming to be a CBI inspector about my bank transactions', 'label': 1},
  {'text': 'A caller threatened to arrest me via video call unless I pay to cancel the FIR', 'label': 1},
  {'text': 'Someone said I am digitally arrested by CBI and must not disconnect or speak to family', 'label': 1},
  {'text': 'CBI officer called demanding I transfer money to government account to avoid arrest', 'label': 1},
  {'text': 'Received a call claiming to be CBI with a warrant against me for money laundering case', 'label': 1},
  {'text': 'Person said my PAN card was misused and CBI is investigating me for financial crime', 'label': 1},
  {'text': 'Caller showed fake CBI ID card on video call and said I must pay to settle the case', 'label': 1},
  {'text': 'Someone impersonating CBI called and said a non-bailable warrant has been issued for me', 'label': 1},
  {'text': 'Called from fake CBI number threatening arrest and asking me to keep it secret from family', 'label': 1},
  {'text': 'CBI caller said to stay on video call for 8 hours as part of digital arrest procedure', 'label': 1},
  {'text': 'Person claiming to be senior CBI officer said my bank account is under investigation', 'label': 1},

  # ── fake_ed (class 2) ────────────────────────────────────────────────────────
  {'text': 'A caller said he is from Enforcement Directorate and my account was used for money laundering', 'label': 2},
  {'text': 'Person claiming to be ED officer called about PMLA case against my bank account', 'label': 2},
  {'text': 'Someone from ED said Rs 2 crore black money transaction was done from my account', 'label': 2},
  {'text': 'Caller said Income Tax department found undeclared foreign exchange in my name', 'label': 2},
  {'text': 'ED officer called saying my account will be seized unless I pay a settlement amount', 'label': 2},
  {'text': 'Person said Enforcement Directorate is investigating tax evasion against me', 'label': 2},
  {'text': 'Got a call claiming ED has found Rs 50 lakh suspicious transaction from my bank', 'label': 2},
  {'text': 'Someone said ED has identified me in a Rs 100 crore money laundering network', 'label': 2},
  {'text': 'Caller said to install AnyDesk for ED verification of my bank account immediately', 'label': 2},
  {'text': 'ED caller asked me to transfer money to a safe government escrow account', 'label': 2},
  {'text': 'Person from Enforcement Directorate demanded I pay penalty to avoid prosecution', 'label': 2},
  {'text': 'Someone said my income tax returns show discrepancies and ED is investigating', 'label': 2},
  {'text': 'Caller claiming to be ED officer sent fake notice via WhatsApp with government seal', 'label': 2},
  {'text': 'Person said ED has evidence of black money linked to my Aadhaar', 'label': 2},
  {'text': 'ED officer called demanding I clear my name by paying Rs 1 lakh to avoid arrest', 'label': 2},

  # ── customs_parcel (class 3) ─────────────────────────────────────────────────
  {'text': 'Got a call saying a parcel in my name was seized at Mumbai airport with drugs inside', 'label': 3},
  {'text': 'Someone claiming to be customs officer called about an intercepted package with my Aadhaar', 'label': 3},
  {'text': 'A caller said customs found foreign currency and SIM cards in a parcel sent in my name', 'label': 3},
  {'text': 'Received call that a DHL package addressed to me contains contraband and I face arrest', 'label': 3},
  {'text': 'Person said FedEx parcel from my address was intercepted with narcotics at customs', 'label': 3},
  {'text': 'Customs officer called demanding clearance fee to release my package seized at airport', 'label': 3},
  {'text': 'Someone said I sent a courier with drugs to another country and it was seized', 'label': 3},
  {'text': 'Caller claimed customs department found illegal goods in parcel with my identity documents', 'label': 3},
  {'text': 'Got call from customs saying a package with fake currency notes has my Aadhaar attached', 'label': 3},
  {'text': 'Person claiming to be airport customs officer said I will be arrested under NDPS Act', 'label': 3},
  {'text': 'Customs caller demanded UPI payment to prevent drug trafficking case against me', 'label': 3},
  {'text': 'Someone said my name appeared in a drug smuggling case at Delhi airport customs', 'label': 3},
  {'text': 'Received WhatsApp message from customs officer about seized parcel in my name', 'label': 3},
  {'text': 'Caller said I need to pay customs duty immediately or criminal case will be filed', 'label': 3},
  {'text': 'Person from customs said a courier with weapons was sent from my mobile number', 'label': 3},

  # ── court_summons (class 4) ──────────────────────────────────────────────────
  {'text': 'Got a call saying I missed a court hearing and a non-bailable warrant has been issued', 'label': 4},
  {'text': 'Person claiming to be a lawyer said I have a court case pending and must pay fine immediately', 'label': 4},
  {'text': 'Received call from someone saying High Court has issued summons against my name', 'label': 4},
  {'text': 'Caller said police will come to my house in 2 hours as court issued arrest warrant', 'label': 4},
  {'text': 'Someone said I am a witness in a fraud case and must appear in court or pay penalty', 'label': 4},
  {'text': 'Person claiming to be court official called demanding immediate payment to avoid contempt', 'label': 4},
  {'text': 'Got WhatsApp message with fake court notice saying I missed a hearing yesterday', 'label': 4},
  {'text': 'Caller said Supreme Court has issued notice for my financial fraud and I should pay bail', 'label': 4},
  {'text': 'Person said I need to pay court fee via UPI to avoid arrest under contempt of court', 'label': 4},
  {'text': 'Received fake court summons via WhatsApp with judge signature asking for settlement', 'label': 4},
  {'text': 'Caller said they are court bailiff and will come to arrest me unless I pay immediately', 'label': 4},
  {'text': 'Person claiming to be advocate said my case is at risk and I should pay urgently', 'label': 4},
  {'text': 'Someone called saying district court has my name in a land fraud case', 'label': 4},
  {'text': 'Caller said I have a case filed against me and must settle before next hearing', 'label': 4},
  {'text': 'Received call claiming to be from court registry about missed summons', 'label': 4},

  # ── trai_suspension (class 5) ────────────────────────────────────────────────
  {'text': 'Got a call saying TRAI will block my mobile number in 2 hours for illegal use', 'label': 5},
  {'text': 'Caller said TRAI found my number was used to make fraudulent calls', 'label': 5},
  {'text': 'Someone claiming to be from TRAI said my SIM will be deactivated for misuse', 'label': 5},
  {'text': 'Received automated call saying press 9 to talk to TRAI officer about number suspension', 'label': 5},
  {'text': 'Person said my Jio number will be permanently blocked unless I verify Aadhaar', 'label': 5},
  {'text': 'TRAI caller said multiple spam complaints have been lodged against my number', 'label': 5},
  {'text': 'Call from TRAI saying my number was used in a cyber fraud and will be blocked', 'label': 5},
  {'text': 'Someone said they are from telecom authority and my number is under investigation', 'label': 5},
  {'text': 'Caller asked me to share OTP to complete KYC re-verification to prevent number block', 'label': 5},
  {'text': 'Person said BSNL is going to blacklist my number permanently due to regulatory violation', 'label': 5},
  {'text': 'Got a call from fake TRAI number demanding money to stop SIM deactivation', 'label': 5},
  {'text': 'Person said my Airtel number was flagged for making fraudulent calls', 'label': 5},
  {'text': 'TRAI caller said my number will be blocked in 2 hours if I do not call back', 'label': 5},
  {'text': 'Automated voice said mobile services will be suspended press 1 to speak to officer', 'label': 5},
  {'text': 'Call claiming TRAI officer found my number linked to a financial crime network', 'label': 5},

  # ── rbi_freeze (class 6) ─────────────────────────────────────────────────────
  {'text': 'Got a call claiming to be RBI saying my account will be frozen for suspicious activity', 'label': 6},
  {'text': 'Person from Reserve Bank of India called saying my account has been flagged', 'label': 6},
  {'text': 'RBI caller said my bank account received suspicious RTGS transfers and will be blocked', 'label': 6},
  {'text': 'Someone from RBI demanded I share OTP to complete account verification', 'label': 6},
  {'text': 'Caller said my SBI account will be permanently blocked unless I verify immediately', 'label': 6},
  {'text': 'Person claiming to be RBI officer asked me to transfer money to a safe escrow account', 'label': 6},
  {'text': 'RBI person called saying my net banking will be suspended for security reasons', 'label': 6},
  {'text': 'Got call from someone claiming my bank account was used for fraudulent transactions', 'label': 6},
  {'text': 'Person said RBI investigation found my HDFC account linked to money laundering', 'label': 6},
  {'text': 'Caller asked me to install AnyDesk for RBI remote verification of my account', 'label': 6},
  {'text': 'Someone from bank said my account has suspicious activity and I should share PIN', 'label': 6},
  {'text': 'RBI caller said my debit card will be blocked and I need to share card details', 'label': 6},
  {'text': 'Person saying RBI cybercell found my account in a financial fraud network', 'label': 6},
  {'text': 'Got fake RBI SMS saying account will be frozen click here to verify', 'label': 6},
  {'text': 'Call from someone claiming to be bank manager asking for my account password', 'label': 6},

  # ── narcotics_bureau (class 7) ───────────────────────────────────────────────
  {'text': 'Caller said NCB found my name in a drug trafficking network and will arrest me', 'label': 7},
  {'text': 'Person claiming to be Narcotics Control Bureau officer said I am part of drug case', 'label': 7},
  {'text': 'Someone said my name was found on a drug dealer contacted list being investigated', 'label': 7},
  {'text': 'NCB caller said my bank account was used to pay drug suppliers', 'label': 7},
  {'text': 'Received call from fake NCB officer saying cocaine was sent using my Aadhaar', 'label': 7},
  {'text': 'Person said they are Anti Narcotics Cell and found my number in a drug gang database', 'label': 7},
  {'text': 'Caller claimed to be NCB inspector and asked me to pay to remove my name from case', 'label': 7},
  {'text': 'Someone said heroin consignment with my identity was seized and I face NDPS charges', 'label': 7},
  {'text': 'NCB officer called saying they have evidence of my involvement in drug supply chain', 'label': 7},
  {'text': 'Person said to keep this call secret as NCB is investigating me for narcotics', 'label': 7},
  {'text': 'Caller said NCB Mumbai found my phone number in communication of drug traffickers', 'label': 7},
  {'text': 'Someone said my WhatsApp was used to coordinate drug delivery and NCB is tracking', 'label': 7},
  {'text': 'NCB caller said they will arrest me under NDPS Act unless I cooperate and pay fee', 'label': 7},
  {'text': 'Person from anti-narcotics department said my car was used in drug transport', 'label': 7},
  {'text': 'Call from fake NCB officer threatening to register case unless I pay immediately', 'label': 7},

  # ── courier_scam (class 8) ───────────────────────────────────────────────────
  {'text': 'Got a call about an undelivered package I never ordered asking me to pay redelivery fee', 'label': 8},
  {'text': 'Someone claiming to be Amazon delivery agent asked for UPI payment to deliver package', 'label': 8},
  {'text': 'Received SMS saying my Flipkart order failed to deliver click link to reschedule', 'label': 8},
  {'text': 'Caller said I won a lucky prize and courier will deliver it but I need to pay customs', 'label': 8},
  {'text': 'Person from fake courier company called about package in my name needing KYC', 'label': 8},
  {'text': 'Got WhatsApp message from DTDC saying package needs payment to release from warehouse', 'label': 8},
  {'text': 'Caller said to click a phishing link to track and redeliver my pending package', 'label': 8},
  {'text': 'Someone said courier package with my address will be returned unless I pay immediately', 'label': 8},
  {'text': 'Received fake India Post SMS saying parcel needs additional fee to be delivered', 'label': 8},
  {'text': 'Caller said they are from Meesho and my order needs re-KYC before delivery', 'label': 8},
  {'text': 'Person said my gift parcel from a relative is stuck at customs and needs clearance fee', 'label': 8},
  {'text': 'Got call from Blue Dart saying my package contains restricted item and needs payment', 'label': 8},
  {'text': 'Someone said my lucky draw prize will be delivered after paying processing fee', 'label': 8},
  {'text': 'Courier agent called asking me to share OTP to confirm my identity for delivery', 'label': 8},
  {'text': 'Received message that Delhivery attempted delivery and I should pay to reschedule', 'label': 8},
]

# Verify
from collections import Counter
label_counts = Counter(d['label'] for d in RAW_DATA)
print(f'Total samples: {len(RAW_DATA)}')
CLASS_NAMES = ['legitimate', 'fake_cbi', 'fake_ed', 'customs_parcel', 'court_summons',
               'trai_suspension', 'rbi_freeze', 'narcotics_bureau', 'courier_scam']
for i, name in enumerate(CLASS_NAMES):
    print(f'  Class {i} ({name}): {label_counts[i]} samples')

In [ ]:
# ── Cell 3: Expand Data with Paraphrasing ─────────────────────────────────────
# Triple the dataset by creating simple paraphrases
import random

def paraphrase(text):
    """Simple rule-based paraphrasing."""
    replacements = [
        ('called', 'phoned'), ('called', 'rang me'),
        ('said', 'told me'), ('said', 'claimed'),
        ('my account', 'my bank account'),
        ('person', 'individual'), ('caller', 'the person'),
        ('immediately', 'right away'), ('immediately', 'urgently'),
        ('money', 'funds'), ('pay', 'transfer'),
        ('arrest', 'detain'), ('someone', 'a person'),
    ]
    result = text
    for orig, replacement in random.sample(replacements, k=min(3, len(replacements))):
        if orig in result.lower():
            result = result.replace(orig, replacement, 1)
    return result

expanded = list(RAW_DATA)  # original
for item in RAW_DATA:
    expanded.append({'text': paraphrase(item['text']), 'label': item['label']})
    expanded.append({'text': paraphrase(paraphrase(item['text'])), 'label': item['label']})

random.shuffle(expanded)
print(f'Expanded dataset: {len(expanded)} samples ({len(RAW_DATA)} original + {len(expanded)-len(RAW_DATA)} paraphrased)')

In [ ]:
# ── Cell 4: Tokenize and Create Dataset ───────────────────────────────────────
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-multilingual-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

# Create HuggingFace dataset
hf_data = Dataset.from_list(expanded)
hf_data = hf_data.map(tokenize, batched=True)
hf_data = hf_data.rename_column('label', 'labels')
hf_data.set_format('torch')

# 80/20 train/test split
split = hf_data.train_test_split(test_size=0.2, seed=42)
train_ds = split['train']
test_ds  = split['test']

print(f'Train: {len(train_ds)} | Test: {len(test_ds)}')

In [ ]:
# ── Cell 5: Train with HuggingFace Trainer ────────────────────────────────────
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=9,
    id2label={i: name for i, name in enumerate(CLASS_NAMES)},
    label2id={name: i for i, name in enumerate(CLASS_NAMES)},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro', zero_division=0),
        'f1_weighted': f1_score(labels, predictions, average='weighted', zero_division=0),
    }

training_args = TrainingArguments(
    output_dir='/content/scam_distilbert',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=50,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=20,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print('Training DistilBERT... (this takes ~20-30 minutes on T4 GPU)')
trainer.train()

In [ ]:
# ── Cell 6: Evaluate ──────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

print('Evaluating on test set...')
predictions = trainer.predict(test_ds)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print('\n' + '='*60)
print('CLARION ScamRadar — Evaluation Results')
print('='*60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

# False Positive Rate on legitimate class
cm = confusion_matrix(y_true, y_pred)
legit_fp = cm[0][1:].sum()  # legitimate predicted as scam
legit_total = cm[0].sum()
fpr = legit_fp / legit_total if legit_total > 0 else 0
print(f'False Positive Rate (legitimate → any scam): {fpr:.1%}')
print(f'Target: <5%  →  {"✅ PASS" if fpr < 0.05 else "❌ FAIL (consider more legitimate training data)"}')

In [ ]:
# ── Cell 7: Save and Download ─────────────────────────────────────────────────
import shutil, os

SAVE_DIR = '/content/scam_distilbert'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Zip for download
ZIP_PATH = '/content/scam_distilbert.zip'
shutil.make_archive('/content/scam_distilbert', 'zip', '/content', 'scam_distilbert')

size_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
print(f'Model saved and zipped. Size: {size_mb:.0f} MB')

from google.colab import files
print('Downloading scam_distilbert.zip...')
files.download(ZIP_PATH)

print('\n' + '='*60)
print('DONE! Next steps:')
print('1. Extract scam_distilbert.zip')
print('2. Move the scam_distilbert/ folder to:')
print('   d:/Projects/ET-AI/CLARION/backend/saved_models/')
print('3. Restart backend — DistilBERT will auto-load')
print('='*60)